TRANSFORMATION & FEATURE ENGINEERING

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "name": ["Asha", "Ram", "Sita", "Hari", "Mina"],
    "age": [22, 35, 28, 45, 31],
    "salary": [20000, 35000, 28000, 80000, 40000],
    "city": ["Kathmandu", "Pokhara", "Kathmandu",
             "Lalitpur", "Pokhara"],
    "purchases": [2, 5, 3, 20, 7]
})

print(df)

 Apply functions

In [ ]:
#Apply a function to one column
df["salary_after_bonus"] = df["salary"].apply(
    lambda x: x * 1.10
)

#Apply a function to every column
df[["age", "salary", "purchases"]].apply(
    np.mean, axis=0
)


#Transform a Series---DataFrame.map() applies a function element by element across a DataFrame.
df["city_upper"] = df["city"].map(
    lambda x: x.upper()
)


# Transform data while preserving the original shape of the input for typical elementwise and same-shape transformations.
df["salary_doubled"] = df["salary"].transform(
    lambda x: x * 2
)


#Pass an entire DataFrame through a function. This is useful for organizing a multi-step data-processing pipeline.
def add_bonus(data):
    return data.assign(
        salary_bonus=data["salary"] * 1.10
    )

def add_age_group(data):
    return data.assign(
        age_group=np.where(data["age"] >= 30,
                           "30_plus", "under_30")
    )

result = (
    df.pipe(add_bonus)
      .pipe(add_age_group)
)

# Evaluate expressions involving DataFrame columns.
#This creates a new column directly from the expression.
df.eval("salary_per_purchase = salary / purchases",
        inplace=True)

#Create new columns or replace existing columns while returning a DataFrame.
result = df.assign(
    annual_salary=df["salary"] * 12,
    double_purchases=df["purchases"] * 2
)
print(result)



Fix skew in numerical data

In [ ]:
# Apply the natural logarithm of 1+ x
df["purchases_log"] = np.log1p(df["purchases"])


# Apply the square root to reduce the effect of large values.
df["purchases_sqrt"] = np.sqrt(df["purchases"])

# Find a power transformation that can make positive data more symmetric.
from scipy import stats

df["salary_boxcox"], fitted_lambda = stats.boxcox(
    df["salary"]
)

print("Lambda:", fitted_lambda)

Scaling

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

df = pd.DataFrame({
    "age":    [20, 30, 40, 50, 60],
    "salary": [20000, 40000, 60000, 80000, 100000]
})

# MinMax
mm = MinMaxScaler()
print(mm.fit_transform(df))

# Standard
ss = StandardScaler()
print(ss.fit_transform(df))

**Binning**

Binning converts a continuous numeric column (age, salary, marks) into discrete categories (young/adult/senior). It’s useful for simplifying noisy data, creating categorical features, and handling outliers.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "age": [5, 12, 18, 22, 25, 31, 38, 45, 52, 67, 80, 95]
})

#pd.cut
bins   = [0, 18, 35, 60, 100]
labels = ["Child", "Young Adult", "Adult", "Senior"]

df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels)
print(df)


#pd.qcut
df["age_quartile"] = pd.qcut(df["age"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
print(df["age_quartile"].value_counts())

